# Cryptocurrency Lead-Lag Statistical Arbitrage

**Author:** Wayne Kirk Schmidt  
**Email:** wayne.kirk.schmidt@gmail.com

---

## Research Motivation

Cryptocurrency markets exhibit strong cross-asset relationships due to shared liquidity pools, overlapping participant bases, and correlated macro drivers. However, price movements across assets are not always simultaneous.

Empirical observation suggests that **large movements in one asset propagate through the market with a short delay**, rather than occurring instantaneously. This raises the possibility that **lead-lag relationships exist between major cryptocurrencies** -- and if such relationships are measurable and persistent, they may provide a basis for statistical arbitrage strategies.

---

## Research Hypothesis

> Conditional on a large return shock in asset A at time t, the expected return of asset B at time t + k is non-zero for small k.

Testing this requires:

- Synchronized historical price data across assets
- Event-based shock detection
- Cross-asset lag measurement and characterization
- Validation via realistic backtesting

---

## Research Pipeline

The analysis is implemented as a staged notebook pipeline. Each stage produces artifacts consumed by the next, ensuring a **modular, reproducible, and auditable workflow**.

| Stage | Notebook | Description |
|:------|:---------|:------------|
| 000 | `000_overview.ipynb` | Structure, hypothesis, and design (this document) |
| 001 | `notebooks/001_download.ipynb` | Data acquisition -- Binance OHLCV for 9 crypto pairs |
| 002 | `notebooks/002_enrich.ipynb` | Feature engineering -- returns, rolling volatility, z-scores |
| 003 | `notebooks/003_analysis.ipynb` | Shock detection and cross-asset lag analysis |
| 003a | `notebooks/003a_regime_classification.ipynb` | Regime segmentation, ADSR characterization, tradability filter |
| 004 | `notebooks/004_strategy.ipynb` | Signal construction and trade mapping |
| 005 | `notebooks/005_backtest.ipynb` | Backtesting, stress testing, and statistical validation |
| 006 | `006_writeup.ipynb` | Findings, conclusions, and future directions |

---

## Stage Definitions

### Stage 1 -- Data Acquisition (`001_download`)

**Reads from:** Binance public API  
**Writes to:** `output/001_download/`

Artifacts: `PRICES.pkl`, `<COIN>.event_panel.pkl`

- Defines the cryptocurrency universe (BTC, ETH, SOL, BNB, ADA, AVAX, XRP, DOGE, LTC)
- Downloads daily OHLCV price data
- Aligns time series across all assets
- Computes daily returns and constructs event panels

---

### Stage 2 -- Data Enrichment (`002_enrich`)

**Reads from:** `output/001_download/`  
**Writes to:** `output/002_enrich/`

Artifacts: `returns_full.pkl`, `rolling_sigma.pkl`, `z_scores.pkl`, `price_wide.pkl`

- Computes rolling volatility (sigma) for each asset
- Normalizes return signals as z-scores
- Prepares enriched datasets for event analysis

---

### Stage 3 -- Analysis (`003_analysis`)

**Reads from:** `output/002_enrich/`  
**Writes to:** `output/003_analysis/`

Artifacts: `shock_events.pkl`, `extreme_z_scores.pkl`, `event_bitmap.pkl`, `sigma_event_matrix.pkl`

- Identifies extreme shock events where |z-score| exceeds threshold
- Visualizes lagged cross-asset responses
- Characterizes the leader-follower relationship structure

---

### Stage 4 -- Strategy Construction (`004_strategy`)

**Reads from:** `output/002_enrich/`, `output/003_analysis/`  
**Writes to:** `output/004_strategy/`

Artifacts: `event_response_df.pkl`, `event_response_expanded.pkl`, `trade_df.pkl`, `daily_returns.pkl`

- Converts observed lead-lag relationships into directional trade signals
- Defines trade construction logic (entry/exit, signal direction)
- Structures candidate trades for backtesting

---

### Stage 5 -- Backtest (`005_backtest`)

**Reads from:** `output/004_strategy/`  
**Writes to:** `output/005_backtest/`

Artifacts: `trade_df.pkl`, `daily_returns.pkl`, `equity_curve.pkl`, `grid_results.pkl`, `performance_summary.pkl`

- Simulates trade execution across a stress grid of execution delays and transaction costs
- Computes performance metrics: Sharpe ratio, win rate, drawdown
- **Statistical significance:** t-test on daily returns -> t = 2.56, p = 0.011
- **Sharpe confidence interval (Lo 2002):** 95% CI = [2.67, 3.86]
- **Walk-forward robustness:** Sharpe across 3 non-overlapping folds ≈ 4.43, 3.07, 2.66

---

### Stage 3a -- Regime Classification (`003a_regime_classification`)

**Reads from:** `output/002_enrich/`, `output/003_analysis/`  
**Writes to:** `output/003a_regimes/`

Artifacts: `regime_series.pkl`, `adsr_profiles.pkl`, `regime_correlations.pkl`, `tradable_signals.pkl`, `context_signals.pkl`, `dragons.pkl`

- Classifies every trading day into Bull / Bear / Transition / Dragon
- Characterizes ADSR profiles per shock type per regime
- Maps cross-asset correlations within each regime
- Applies the tradability filter to separate signal from context
- Documents Dragon events explicitly

---

### Stage 6 -- Writeup (`006_writeup`)

**Reads from:** `output/005_backtest/`  
**Produces:** Final research narrative

- Interprets backtest results against the original hypothesis
- Evaluates statistical robustness
- Documents limitations and proposes future directions

---

## Scope

**In scope:**
- Major cryptocurrency assets on daily time resolution
- Short-term lead-lag relationships (t+0, t+1 execution)
- Statistical validation under realistic transaction cost assumptions

**Out of scope:**
- Intraday execution modeling
- Production deployment or live trading
- Portfolio-level optimization across multiple simultaneous strategies

---

## Design Principles

1. **Determinism** -- each stage produces well-defined artifacts with no hidden dependencies
2. **Modularity** -- each stage can be executed and validated independently
3. **Reproducibility** -- all outputs are derived from persisted artifacts
4. **Transparency** -- all statistical assumptions are stated and tested explicitly

---

## Summary

This document defines the structure, hypothesis, and execution plan for investigating lead-lag relationships in cryptocurrency markets. The pipeline implements, tests, and evaluates this hypothesis in a controlled and reproducible manner -- culminating in a validated statistical arbitrage strategy with strong and significant risk-adjusted returns.
